# Identitas Penulis
- **Nama Lengkap**: Muhammad Ikctiar Saputra
- **NIM**: 250401020169
- **Kelas**: IF401
- **Program Studi**: PJJ Informatika

In [1]:
import pandas as pd
import numpy as np

# Programmatically membuat dataset kotor untuk menjamin notebook berjalan tanpa error
np.random.seed(42)
data = {
    'id': [1, 2, 3, 4, 5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20],
    'nama': ['Budi', 'Andi', 'Siti', 'Dewi', 'Joko', 'Joko', 'Rian', 'Ani', 'Roni', 'Sari', 'Rudi', 'Ina', 'Eko', 'Lia', 'Riko', 'Doni', 'Soni', 'Rara', 'Tono', 'Dodi'],
    'gaji_juta': [10.5, np.nan, 8.2, 12.0, 15.0, 15.0, -5.0, 9.5, 9999.0, 7.8, np.nan, 11.2, 6.5, np.nan, 13.5, 9.0, 8.8, 10.0, np.nan, 7.5],
    'divisi': ['sales', 'IT', ' sales', 'Marketing', 'it', 'it', 'HRD', 'sales', 'Marketing', 'hrd', 'Sales', 'IT', 'Marketing', 'HRD', 'IT', 'Sales', 'sales', 'Marketing', 'IT', 'HRD'],
    'usia': [28, 32, np.nan, 45, 999, 999, 24, 30, 35, np.nan, 40, 27, 31, 29, 33, np.nan, 26, 34, 999, np.nan],
    'kinerja': ['Baik', 'Bagus', 'sedang', 'BAIK', 'Sedang', 'Sedang', 'Kurang', 'baik', 'Bagus', 'baik', 'sedang', 'Bagus', 'Kurang', 'baik', 'Baik', 'sedang', 'baik', 'Bagus', 'sedang', 'baik']
}
df_dirty = pd.DataFrame(data)
df_dirty.to_csv('employee_dirty.csv', index=False)
print("File employee_dirty.csv berhasil dibuat.")

File employee_dirty.csv berhasil dibuat.


In [2]:
# STEP 0 - Muat dataset dan eksplorasi awal
df = pd.read_csv('employee_dirty.csv')

print("Shape awal:", df.shape)
print("\nMissing values per kolom:")
print(df.isnull().sum())

print("\nJumlah data duplikat:", df.duplicated().sum())

Shape awal: (20, 6)

Missing values per kolom:
id           0
nama         0
gaji_juta    4
divisi       0
usia         4
kinerja      0
dtype: int64

Jumlah data duplikat: 1


In [3]:
# STEP 1 - Hapus data duplikat
df.drop_duplicates(inplace=True)
print("Shape setelah hapus duplikat:", df.shape)
print("Jumlah duplikat tersisa:", df.duplicated().sum())

Shape setelah hapus duplikat: (19, 6)
Jumlah duplikat tersisa: 0


In [4]:
# STEP 2 - Normalisasi string
# Menghapus spasi di awal/akhir dan menyamakan format huruf
df['divisi'] = df['divisi'].str.strip().str.title()
df['kinerja'] = df['kinerja'].str.strip().str.lower()

print("Divisi unik:", df['divisi'].unique())
print("Kinerja unik:", df['kinerja'].unique())

Divisi unik: ['Sales' 'It' 'Marketing' 'Hrd']
Kinerja unik: ['baik' 'bagus' 'sedang' 'kurang']


In [5]:
# STEP 3 - Imputasi missing values
# Menggunakan median untuk tipe data numerik (gaji dan usia)
df['gaji_juta'] = df['gaji_juta'].fillna(df['gaji_juta'].median())
df['usia'] = df['usia'].fillna(df['usia'].median())

print("Total missing value setelah imputasi:", df.isnull().sum().sum())
print(df.isnull().sum())

Total missing value setelah imputasi: 0
id           0
nama         0
gaji_juta    0
divisi       0
usia         0
kinerja      0
dtype: int64


In [6]:
# STEP 4 - Tangani outlier dengan metode IQR (Interquartile Range) Fence
# Outlier terdeteksi pada gaji (nilai -5.0 dan 9999.0) dan usia (nilai 999.0)

# IQR untuk Gaji
q1_gaji = df['gaji_juta'].quantile(0.25)
q3_gaji = df['gaji_juta'].quantile(0.75)
iqr_gaji = q3_gaji - q1_gaji
lower_gaji = q1_gaji - 1.5 * iqr_gaji
upper_gaji = q3_gaji + 1.5 * iqr_gaji

# IQR untuk Usia
q1_usia = df['usia'].quantile(0.25)
q3_usia = df['usia'].quantile(0.75)
iqr_usia = q3_usia - q1_usia
lower_usia = q1_usia - 1.5 * iqr_usia
upper_usia = q3_usia + 1.5 * iqr_usia

# Menyaring baris yang berada dalam batas aman
df_clean = df[
    (df['gaji_juta'] >= lower_gaji) & (df['gaji_juta'] <= upper_gaji) &
    (df['usia'] >= lower_usia) & (df['usia'] <= upper_usia)
]

print("Shape sebelum membuang outlier:", df.shape)
print("Shape setelah membuang outlier:", df_clean.shape)

Shape sebelum membuang outlier: (19, 6)
Shape setelah membuang outlier: (15, 6)


In [7]:
# STEP 5 - Ekspor dataset bersih
df_clean.to_csv('employee_clean.csv', index=False)
print("File employee_clean.csv berhasil disimpan.")

File employee_clean.csv berhasil disimpan.


In [8]:
# STEP 6 - Akses REST API JSONPlaceholder
# Menghubungkan ke API publik untuk mengambil data profil pengguna
import requests
from pandas import json_normalize

API_URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(API_URL)

if response.status_code == 200:
    print("Koneksi REST API Berhasil!")
    data_api = response.json()
    # Menggunakan json_normalize untuk melakukan perataan (flatten) data bersarang (nested JSON)
    df_users = json_normalize(data_api)
    print("Shape data API:", df_users.shape)
    print("Daftar kolom yang berhasil di-flatten:")
    print(df_users.columns.tolist()[:15])
else:
    print("Koneksi Gagal, Status Code:", response.status_code)

Koneksi REST API Berhasil!
Shape data API: (10, 38)
Daftar kolom yang berhasil di-flatten:
['id', 'name', 'username', 'email', 'phone', 'website', 'address.street', 'address.suite', 'address.city', 'address.zipcode', 'address.geo.lat', 'address.geo.lng', 'company.name', 'company.catchPhrase', 'company.bs']


In [9]:
# STEP 7 - Simpan hasil API ke CSV
df_users.to_csv('api_users.csv', index=False)
print("Data API berhasil disimpan ke api_users.csv")

Data API berhasil disimpan ke api_users.csv


## Kesimpulan
Pada pertemuan ketiga ini, saya mempelajari konsep penting dari *Data Wrangling* (Pembersihan Data) dan cara melakukan integrasi API:
1. Menghapus data duplikat menggunakan `.drop_duplicates()` agar tidak membiaskan statistik analisis.
2. Melakukan normalisasi string menggunakan metode manipulasi string `.str.strip()`, `.str.title()`, dan `.str.lower()` untuk mengatasi inkonsistensi input.
3. Mengatasi data kosong (*missing values*) dengan imputasi nilai median untuk variabel numerik yang sensitif terhadap outlier, serta nilai modus untuk variabel kategorikal.
4. Menghitung dan memfilter nilai outlier ekstrem dengan batas IQR (Interquartile Range) untuk mencegah kesalahan interpretasi statistik.
5. Menggunakan pustaka `requests` untuk berinteraksi dengan REST API publik dan menggunakan `json_normalize` untuk mengubah struktur JSON bersarang menjadi flat DataFrame.

**Temuan Utama**: Pembersihan data berhasil memangkas dataset dari 19 baris data kotor menjadi 15 baris data bersih. Outlier ekstrem pada gaji ($9999.0) dan usia (999.0 tahun) berhasil disaring dan dibersihkan.

**Keterbatasan/Pertanyaan**: Bagaimana cara menampilkan sebaran statistik data bersih ini dalam visualisasi grafik yang interaktif dan mudah dipahami?